In [ ]:
import json
import gc
import warnings

import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

warnings.filterwarnings("ignore")
pd.options.display.precision = 15


In [ ]:
try:
    import google.colab
    IS_COLAB = True
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    IS_COLAB = False

ROOT = Path("/content/drive/MyDrive/minor-thesis") if IS_COLAB else Path.cwd()
DATASET_PATH = ROOT / "dataset"
SAVED_PATH = ROOT / "saved"
SAVED_PATH.mkdir(parents=True, exist_ok=True)
RANDOM_SEED = 42
print(f"Running on {'Google Colab' if IS_COLAB else 'Local'}")
print(f"Dataset path: {DATASET_PATH}")


In [ ]:
train = pd.read_parquet(DATASET_PATH / "merged_train.parquet")
print(f"train {train.shape}")


In [ ]:
def unwrap_estimator(model):
    if hasattr(model, "named_steps"):
        return list(model.named_steps.values())[-1]
    if hasattr(model, "estimator"):
        return model.estimator
    return model


def top_feature_importances(model, columns, n=20):
    est = unwrap_estimator(model)
    if hasattr(est, "feature_importances_"):
        importances = np.asarray(est.feature_importances_, dtype=float)
    elif hasattr(est, "coef_"):
        importances = np.abs(np.asarray(est.coef_, dtype=float).ravel())
    else:
        return []
    if importances.size != len(columns):
        return []
    total = float(importances.sum())
    n = min(int(n), len(importances))
    order = np.argsort(importances)[::-1][:n]
    rows = []
    for rank, idx in enumerate(order, start=1):
        value = float(importances[idx])
        pct = (value / total * 100.0) if total > 0 else 0.0
        rows.append(
            {
                "rank": int(rank),
                "feature": str(columns[idx]),
                "importance": value,
                "importance_pct": pct,
            }
        )
    return rows


def positive_scores(model, X):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    if hasattr(model, "decision_function"):
        return model.decision_function(X)
    return model.predict(X).astype(float)


def evaluate(model, X_train, X_valid, y_train, y_valid, name):
    print(name)
    print(f"Features: {X_train.shape[1]}  train rows: {len(X_train):,}")
    model.fit(X_train, y_train)
    y_pred = model.predict(X_valid)
    y_score = positive_scores(model, X_valid)
    accuracy = accuracy_score(y_valid, y_pred)
    precision = precision_score(y_valid, y_pred, zero_division=0)
    recall = recall_score(y_valid, y_pred, zero_division=0)
    f1 = f1_score(y_valid, y_pred, zero_division=0)
    roc_auc = roc_auc_score(y_valid, y_score)
    pr_auc = average_precision_score(y_valid, y_score)
    balanced_acc = balanced_accuracy_score(y_valid, y_pred)
    mcc = matthews_corrcoef(y_valid, y_pred)
    cm = confusion_matrix(y_valid, y_pred)
    print(f"Accuracy           : {accuracy:.4f}")
    print(f"Precision          : {precision:.4f}")
    print(f"Recall             : {recall:.4f}")
    print(f"F1 Score           : {f1:.4f}")
    print(f"ROC-AUC            : {roc_auc:.4f}")
    print(f"PR-AUC             : {pr_auc:.4f}")
    print(f"Balanced Accuracy  : {balanced_acc:.4f}")
    print(f"MCC                : {mcc:.4f}")
    print("Confusion Matrix:")
    print(cm)
    print(
        classification_report(
            y_valid, y_pred, target_names=["Legitimate", "Fraud"], digits=4, zero_division=0
        )
    )
    top20 = top_feature_importances(model, X_train.columns)
    gc.collect()
    return {
        "Model": name,
        "Features": int(X_train.shape[1]),
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "ROC-AUC": roc_auc,
        "PR-AUC": pr_auc,
        "Balanced Accuracy": balanced_acc,
        "MCC": mcc,
        "TN": int(cm[0, 0]),
        "FP": int(cm[0, 1]),
        "FN": int(cm[1, 0]),
        "TP": int(cm[1, 1]),
        "Top20Importances": json.dumps(top20),
    }


In [ ]:
def _scale_pos_weight(y):
    n_neg = int((y == 0).sum())
    n_pos = max(int((y == 1).sum()), 1)
    return n_neg / n_pos


def build_model(model_type, params, y_train, n_jobs=-1):
    # Same constructors as 02_ml_models_fit.ipynb Optuna refit.
    p = dict(params)
    if model_type == "LogisticRegression":
        return Pipeline(
            [
                ("scaler", StandardScaler()),
                (
                    "clf",
                    LogisticRegression(
                        class_weight="balanced",
                        max_iter=2000,
                        solver="lbfgs",
                        random_state=RANDOM_SEED,
                        **p,
                    ),
                ),
            ]
        )
    if model_type == "DecisionTree":
        return DecisionTreeClassifier(
            class_weight="balanced", random_state=RANDOM_SEED, **p
        )
    if model_type == "RandomForest":
        return RandomForestClassifier(
            class_weight="balanced", random_state=RANDOM_SEED, n_jobs=n_jobs, **p
        )
    if model_type == "LightGBM":
        return LGBMClassifier(
            class_weight="balanced",
            random_state=RANDOM_SEED,
            n_jobs=n_jobs,
            verbosity=-1,
            subsample_freq=1,
            **p,
        )
    if model_type == "XGBoost":
        return XGBClassifier(
            random_state=RANDOM_SEED,
            n_jobs=n_jobs,
            eval_metric="logloss",
            tree_method="hist",
            scale_pos_weight=_scale_pos_weight(y_train),
            **p,
        )
    if model_type == "CatBoost":
        return CatBoostClassifier(
            random_seed=RANDOM_SEED,
            auto_class_weights="Balanced",
            verbose=False,
            allow_writing_files=False,
            thread_count=n_jobs if n_jobs > 0 else -1,
            **p,
        )
    raise ValueError(f"Unknown model_type {model_type}")


def parse_params(value):
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return {}
    if isinstance(value, dict):
        return dict(value)
    return json.loads(value)


In [ ]:
train = train.sort_values("TransactionDT").reset_index(drop=True)
y = train["isFraud"]
split_idx = int(len(train) * 0.8)
y_train = y.iloc[:split_idx]
y_valid = y.iloc[split_idx:]

ID_COLS = ["isFraud", "TransactionID"]
FE_COLS = ["uid", "uid2", "DT_month", "DT_week", "DT_day", "DT_weekday", "DT_hour"]

baseline_cols = [c for c in train.columns if c not in ID_COLS + FE_COLS]
feature_cols = [c for c in train.columns if c not in ID_COLS]
experiments = [
    ("Baseline", train[baseline_cols].iloc[:split_idx], train[baseline_cols].iloc[split_idx:]),
    ("Feature Engineering", train[feature_cols].iloc[:split_idx], train[feature_cols].iloc[split_idx:]),
]

reduced_path = DATASET_PATH / "merged_train_reduced.parquet"
train_reduced = pd.read_parquet(reduced_path).sort_values("TransactionDT").reset_index(drop=True)
if len(train_reduced) != len(train):
    raise ValueError(f"reduced rows {len(train_reduced):,} != full train {len(train):,}")

reduced_baseline_cols = [c for c in train_reduced.columns if c not in ID_COLS + FE_COLS]
reduced_feature_cols = [c for c in train_reduced.columns if c not in ID_COLS]
experiments.extend(
    [
        (
            "Reduced Baseline",
            train_reduced[reduced_baseline_cols].iloc[:split_idx],
            train_reduced[reduced_baseline_cols].iloc[split_idx:],
        ),
        (
            "Reduced Feature Engineering",
            train_reduced[reduced_feature_cols].iloc[:split_idx],
            train_reduced[reduced_feature_cols].iloc[split_idx:],
        ),
    ]
)

for label, X_tr, X_va in experiments:
    print(f"{label}: {X_tr.shape[1]} features  train {len(X_tr):,}  valid {len(X_va):,}")


In [ ]:
# SMOTE oversamples minority to 1:1. RandomUnderSampler drops majority to 1:1.
# Resampling is train-only; holdout stays original.
SAMPLERS = ("SMOTE", "Undersampling")

tuned_path = SAVED_PATH / "ml_results.parquet"
tuned = pd.read_parquet(tuned_path)
tuned = tuned[
    ~tuned["Model"].astype(str).str.contains(r" - (SMOTE|Undersampling)$", regex=True)
].copy()
print(f"Optuna rows loaded: {len(tuned)}")

RESULT_COLS = [
    "Model",
    "ModelType",
    "Dataset",
    "Sampling",
    "Features",
    "Accuracy",
    "Precision",
    "Recall",
    "F1",
    "ROC-AUC",
    "PR-AUC",
    "Balanced Accuracy",
    "MCC",
    "TN",
    "FP",
    "FN",
    "TP",
    "TuneROC-AUC",
    "BestParams",
    "Top20Importances",
]
all_results = []


def resample_train(X, y, method):
    X_arr = np.asarray(X, dtype=np.float32)
    y_arr = np.asarray(y)
    if method == "SMOTE":
        sampler = SMOTE(sampling_strategy=1.0, k_neighbors=5, random_state=RANDOM_SEED)
    elif method == "Undersampling":
        sampler = RandomUnderSampler(sampling_strategy=1.0, random_state=RANDOM_SEED)
    else:
        return X, y
    X_res, y_res = sampler.fit_resample(X_arr, y_arr)
    X_out = pd.DataFrame(X_res, columns=list(X.columns))
    y_out = pd.Series(y_res, name=getattr(y, "name", "isFraud"))
    print(
        f"  {method}: {len(y):,} -> {len(y_out):,}  "
        f"pos={int((y_out == 1).sum()):,}  neg={int((y_out == 0).sum()):,}"
    )
    return X_out, y_out


def existing_models():
    if not tuned_path.exists():
        return set()
    return set(pd.read_parquet(tuned_path)["Model"].astype(str))


def save_results():
    # Append/replace only the SMOTE/undersampling rows; leave Optuna core rows intact.
    if not all_results:
        return
    new_df = pd.DataFrame(all_results).drop_duplicates(subset=["Model"], keep="last")
    new_df = new_df[[c for c in RESULT_COLS if c in new_df.columns]]
    if tuned_path.exists():
        old = pd.read_parquet(tuned_path)
        if "Model" in old.columns:
            old = old[~old["Model"].isin(new_df["Model"])]
        new_df = pd.concat([old, new_df], ignore_index=True, sort=False)
    new_df = new_df.drop_duplicates(subset=["Model"], keep="last")
    new_df.to_parquet(tuned_path, index=False)
    print(f"saved {tuned_path}  n={len(new_df)}")


def params_for(prefix, model_type, dataset):
    name = f"{prefix} - {dataset}"
    hit = tuned[tuned["Model"].astype(str) == name]
    if hit.empty:
        hit = tuned[(tuned["ModelType"] == model_type) & (tuned["Dataset"] == dataset)]
    if hit.empty:
        return None, None
    row = hit.iloc[0]
    return parse_params(row["BestParams"]), row.get("TuneROC-AUC")


def run_family(prefix, model_type):
    done = existing_models()
    for label, X_tr, X_va in experiments:
        params, tune_auc = params_for(prefix, model_type, label)
        if params is None:
            print(f"skip {prefix} - {label}: no Optuna BestParams")
            continue
        for sampling in SAMPLERS:
            name = f"{prefix} - {label} - {sampling}"
            if name in done:
                print(f"skip {name}")
                continue
            X_s, y_s = resample_train(X_tr, y_train, sampling)
            model = build_model(model_type, params, y_s)
            row = evaluate(model, X_s, X_va, y_s, y_valid, name)
            row["ModelType"] = model_type
            row["Dataset"] = label
            row["Sampling"] = sampling
            row["BestParams"] = json.dumps(params)
            row["TuneROC-AUC"] = tune_auc
            all_results.append(row)
            save_results()
            done.add(name)
            del model, X_s, y_s
            gc.collect()


In [ ]:
run_family("Logistic Regression", "LogisticRegression")


In [ ]:
run_family("Decision Tree", "DecisionTree")


In [ ]:
run_family("RF", "RandomForest")


In [ ]:
run_family("LightGBM", "LightGBM")


In [ ]:
run_family("XGBoost", "XGBoost")


In [ ]:
run_family("CatBoost", "CatBoost")
